# 3.1 — Multi-Dataset Regression Modelling (2.1 Manual vs 2.2 UMAP/MICE)

Run the **same regression pipeline** across multiple prepared datasets and models.  
Each **(dataset × model)** pair runs in its own cell so you can execute and inspect them individually.

Datasets:
- **2.1-Manual** → `2.1-train.parquet` / `2.1-test.parquet`
- **2.2-UMAP-MICE** → `2.2-train.parquet` / `2.2-test.parquet`

Models: `LinearRegression`, `Ridge`, `Lasso`, `RandomForest`, `GradientBoosting`, `XGBoost`

## Outline

| # | Section |
|---|---|
| 1 | Imports and global setup |
| 2 | Reusable pipeline factory (single model) |
| 3 | Load all datasets |
| 4 | Model registry & `run_model()` helper |
| 5 | Individual runs — one cell per (dataset × model) |
| 6 | Cross-dataset × cross-model comparison (winner table) |
| 7 | Residual analysis — best pipeline |
| 8 | Feature importance & SHAP — best pipeline |
| 9 | Dummy patient prediction — best pipeline |
| 10 | Save all models and metrics |


## 1. Imports and Global Setup

In [4]:
import os
from pathlib import Path

import pandas as pd
import numpy as np
!pip install polars
import polars as pl

from sklearn.model_selection import RepeatedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

import matplotlib.pyplot as plt
import seaborn as sns

# Optional explainability library
try:
    import shap
except Exception:
    shap = None

pd.options.display.max_columns = 200
pd.options.display.precision = 4

print('Libraries loaded. SHAP available:', shap is not None)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 824.0/824.0 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 MB 33.2 MB/s eta 0:00:00
Libraries loaded. SHAP available: False


## 2. Reusable Pipeline Factory Function

`build_regression_pipeline` handles one **(dataset, model)** pair:

1. Feature / target separation (Polars-native)
2. Column type detection via Polars `.dtype` (handles `uint8`/`uint16` from Parquet)
3. `ColumnTransformer` preprocessor (median imputation → scaling / one-hot)
4. `RepeatedKFold` CV — returns CV metrics
5. Fits model on full training set → returns test metrics

**Pass any sklearn estimator** as the `model` argument.  
Polars DataFrames are only converted to pandas at the sklearn boundary (`.to_pandas()`).


In [ ]:
def build_regression_pipeline(
    train_df:   pl.DataFrame,
    test_df:    pl.DataFrame,
    target:     str,
    model,                          # single sklearn estimator
    model_name: str = 'Model',
    drop_cols:  list = None,
    n_splits:   int  = 5,
    n_repeats:  int  = 3,
    random_state: int = 42,
) -> dict:
    """
    Cross-validate and evaluate ONE sklearn regression model on a Polars dataset.

    Parameters
    ----------
    train_df, test_df : pl.DataFrame
    target            : str          — target column name
    model             : sklearn estimator — pass a model from MODELS registry
    model_name        : str          — label for display / storage
    drop_cols         : list         — extra columns to exclude from features
    n_splits, n_repeats, random_state — RepeatedKFold settings

    Returns
    -------
    dict with keys:
        model_name, pipeline,
        X_train, X_test  (pl.DataFrame),
        y_train, y_test, y_pred  (np.ndarray),
        feature_names, numeric_cols, cat_cols,
        cv_mae, cv_rmse, cv_r2,
        test_mae, test_rmse, test_r2
    """
    # ── 1. Feature / target separation (Polars) ──────────────────────────────
    all_drop     = [c for c in ([target] + (drop_cols or [])) if c in train_df.columns]
    feature_cols = [c for c in train_df.columns if c not in all_drop]

    X_train = train_df.select(feature_cols)
    y_train = train_df[target].to_numpy()
    X_test  = test_df.select([c for c in feature_cols if c in test_df.columns])
    y_test  = test_df[target].to_numpy()

    # ── 2. Column type detection (Polars native) ──────────────────────────────
    _numeric_dtypes = {
        pl.Int8,  pl.Int16,  pl.Int32,  pl.Int64,
        pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64,
        pl.Float32, pl.Float64,
    }
    _string_dtypes = {pl.Utf8, pl.String, pl.Categorical}

    numeric_cols = [c for c in X_train.columns if X_train[c].dtype in _numeric_dtypes]
    cat_cols     = [c for c in X_train.columns if X_train[c].dtype in _string_dtypes]

    print(f'  Features : {len(numeric_cols)} numeric, {len(cat_cols)} categorical')

    # ── 3. sklearn boundary: convert Polars → pandas ──────────────────────────
    X_train_pd = X_train.to_pandas()
    X_test_pd  = X_test.to_pandas()

    # ── 4. Preprocessor ───────────────────────────────────────────────────────
    numeric_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
    ])
    transformers = []
    if numeric_cols:
        transformers.append(('num', numeric_transformer, numeric_cols))
    if cat_cols:
        cat_transformer = Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
            ('onehot',  OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ])
        transformers.append(('cat', cat_transformer, cat_cols))

    if not transformers:
        raise ValueError('No numeric or categorical columns found.')

    preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')

    # ── 5. RepeatedKFold cross-validation (single model) ─────────────────────
    cv      = RepeatedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=random_state)
    scoring = {
        'MAE':  'neg_mean_absolute_error',
        'RMSE': 'neg_root_mean_squared_error',
        'R2':   'r2',
    }
    pipe   = Pipeline([('pre', preprocessor), ('model', model)])
    cv_res = cross_validate(pipe, X_train_pd, y_train, cv=cv, scoring=scoring,
                            n_jobs=-1, return_train_score=False)

    cv_mae  = float(-cv_res['test_MAE'].mean())
    cv_rmse = float(-cv_res['test_RMSE'].mean())
    cv_r2   = float(cv_res['test_R2'].mean())
    print(f'  CV  : MAE={cv_mae:.4f}  RMSE={cv_rmse:.4f}  R²={cv_r2:.4f}')

    # ── 6. Fit on full training set ───────────────────────────────────────────
    pipeline = Pipeline([('pre', preprocessor), ('model', model)])
    pipeline.fit(X_train_pd, y_train)

    # ── 7. Test-set evaluation ────────────────────────────────────────────────
    y_pred    = pipeline.predict(X_test_pd)
    test_mae  = float(mean_absolute_error(y_test, y_pred))
    test_rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
    test_r2   = float(r2_score(y_test, y_pred))
    print(f'  Test: MAE={test_mae:.4f}  RMSE={test_rmse:.4f}  R²={test_r2:.4f}')

    # ── 8. Feature names after one-hot expansion ──────────────────────────────
    onehot_cols = []
    if cat_cols:
        enc = pipeline.named_steps['pre'].named_transformers_['cat'].named_steps['onehot']
        onehot_cols = enc.get_feature_names_out(cat_cols).tolist()
    feature_names = numeric_cols + onehot_cols

    return {
        'model_name':    model_name,
        'pipeline':      pipeline,
        'X_train':       X_train,       # pl.DataFrame
        'X_test':        X_test,        # pl.DataFrame
        'y_train':       y_train,       # np.ndarray
        'y_test':        y_test,        # np.ndarray
        'y_pred':        y_pred,        # np.ndarray
        'feature_names': feature_names,
        'numeric_cols':  numeric_cols,
        'cat_cols':      cat_cols,
        'cv_mae':        cv_mae,   'cv_rmse':   cv_rmse,   'cv_r2':   cv_r2,
        'test_mae':      test_mae, 'test_rmse': test_rmse, 'test_r2': test_r2,
    }

print('build_regression_pipeline() defined.')


build_regression_pipeline() defined.


## 3. Load All Datasets

Add or remove entries in `DATASETS` to compare more pipelines.  
Each entry maps a label to its local train/test parquet paths.


In [ ]:
# ── Dataset registry — add/remove entries freely ─────────────────────────────
DATASETS = {
    '2.1-Manual': {
        'train_path': 'https://raw.githubusercontent.com/LALITAMITTAL18/EAISI-NHS/main/notebooks/Experiment/Knee/data/interim/2.1-train.parquet',
        'test_path':  'https://raw.githubusercontent.com/LALITAMITTAL18/EAISI-NHS/main/notebooks/Experiment/Knee/data/interim/2.1-test.parquet',
    },
    '2.2-UMAP-MICE': {
        'train_path': 'https://raw.githubusercontent.com/LALITAMITTAL18/EAISI-NHS/main/notebooks/Experiment/Knee/data/interim/2.2-train.parquet',
        'test_path':  'https://raw.githubusercontent.com/LALITAMITTAL18/EAISI-NHS/main/notebooks/Experiment/Knee/data/interim/2.1-test.parquet',
    },
}

TARGET = 'health_gain'

# ── Load each dataset as Polars DataFrames ────────────────────────────────────
all_data = {}   # label -> (train_pl, test_pl, drop_cols)

for label, paths in DATASETS.items():
    train_pl  = pl.read_parquet(paths['train_path'])
    test_pl   = pl.read_parquet(paths['test_path'])
    drop_cols = ['OHS_Success'] if 'OHS_Success' in train_pl.columns else []
    all_data[label] = (train_pl, test_pl, drop_cols)
    print(f'{label:20s}  train={train_pl.shape}  test={test_pl.shape}  drop={drop_cols}')

print(f'\nTarget : {TARGET}')
print(f'Loaded : {list(all_data.keys())}')


2.1-Manual            train=(94550, 46)  test=(23638, 46)  extra_drop=[]
2.2-UMAP-MICE         train=(111388, 47)  test=(23638, 46)  extra_drop=[]

Target: health_gain
Datasets loaded: ['2.1-Manual', '2.2-UMAP-MICE']


## 4. Model Registry & Runner

`MODELS` is the single source of truth for all candidate estimators, including XGBoost.  
`run_model(dataset_label, model_name)` runs one pair and stores the result — call it in any cell below.


In [ ]:
# ── Model registry ────────────────────────────────────────────────────────────
MODELS = {
    'LinearRegression': LinearRegression(),
    'Ridge':            Ridge(alpha=1.0, random_state=42),
    'Lasso':            Lasso(alpha=0.01, random_state=42, max_iter=10_000),
    'RandomForest':     RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    'GradientBoosting': GradientBoostingRegressor(n_estimators=300, random_state=42),
}
if _xgb_available:
    MODELS['XGBoost'] = XGBRegressor(
        n_estimators=300, learning_rate=0.05, subsample=0.8,
        random_state=42, n_jobs=-1, verbosity=0,
    )

print(f'Models available: {list(MODELS.keys())}')

# ── Results store: { dataset_label : { model_name : result_dict } } ──────────
all_results = {}

# ── Convenience runner ────────────────────────────────────────────────────────
def run_model(dataset_label: str, model_name: str) -> dict:
    """Run one (dataset × model) combination and cache result in all_results."""
    if model_name not in MODELS:
        raise ValueError(f"Unknown model '{model_name}'. Available: {list(MODELS.keys())}")
    train_pl, test_pl, drop_cols = all_data[dataset_label]
    print(f'▶  {dataset_label}  ×  {model_name}')
    result = build_regression_pipeline(
        train_pl, test_pl, target=TARGET,
        model=MODELS[model_name], model_name=model_name,
        drop_cols=drop_cols,
    )
    all_results.setdefault(dataset_label, {})[model_name] = result
    return result

print('run_model() ready — call run_model(dataset_label, model_name) in each cell below.')



  Dataset: 2.1-Manual
Numeric features     : 44
Categorical features : 1
  numeric dtypes     : ['Float32', 'Float64', 'Int32', 'UInt32', 'UInt8']
  LinearRegression     RMSE=8.0894  R²=0.3151
  Ridge                RMSE=8.0862  R²=0.3156
  Lasso                RMSE=8.0860  R²=0.3156


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


KeyboardInterrupt: 

## 5. Individual Runs — Dataset × Model

Each cell below runs **one (dataset, model) pair** independently.  
Run them selectively or all at once. Results accumulate in `all_results`.

> **Tip:** To add a new model, add it to `MODELS` in the cell above, then add a `run_model(...)` cell here.


### Dataset: 2.1-Manual


In [ ]:
run_model('2.1-Manual', 'LinearRegression')


In [ ]:
run_model('2.1-Manual', 'Ridge')


In [ ]:
run_model('2.1-Manual', 'Lasso')


In [ ]:
run_model('2.1-Manual', 'RandomForest')


In [ ]:
run_model('2.1-Manual', 'GradientBoosting')


In [ ]:
run_model('2.1-Manual', 'XGBoost')


### Dataset: 2.2-UMAP-MICE


In [ ]:
run_model('2.2-UMAP-MICE', 'LinearRegression')


In [ ]:
run_model('2.2-UMAP-MICE', 'Ridge')


In [ ]:
run_model('2.2-UMAP-MICE', 'Lasso')


In [ ]:
run_model('2.2-UMAP-MICE', 'RandomForest')


In [ ]:
run_model('2.2-UMAP-MICE', 'GradientBoosting')


In [ ]:
run_model('2.2-UMAP-MICE', 'XGBoost')


## 6. Compare All Results

Collect every (dataset × model) result stored in `all_results` and find the overall winner.  
The best row (lowest Test RMSE) is highlighted in green — downstream sections use that pipeline.


In [ ]:
# ── Flatten all_results into a comparison table ───────────────────────────────
comparison_rows = []
for ds_label, models_dict in all_results.items():
    for mdl_name, res in models_dict.items():
        comparison_rows.append({
            'Dataset':   ds_label,
            'Model':     mdl_name,
            'CV RMSE':   round(res['cv_rmse'],   4),
            'CV R²':     round(res['cv_r2'],      4),
            'Test MAE':  round(res['test_mae'],   4),
            'Test RMSE': round(res['test_rmse'],  4),
            'Test R²':   round(res['test_r2'],    4),
        })

comparison_df = (
    pd.DataFrame(comparison_rows)
    .sort_values('Test RMSE')
    .reset_index(drop=True)
)

def highlight_winner(s):
    return ['background-color: #d4edda; font-weight: bold'
            if i == 0 else '' for i in range(len(s))]

print('=== Full Comparison: Dataset × Model ===\n')
display(comparison_df.style.apply(highlight_winner, axis=0))

# ── Bar charts ────────────────────────────────────────────────────────────────
comparison_df['Label'] = comparison_df['Dataset'] + '\n' + comparison_df['Model']

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
fig.suptitle('Test-Set Performance — All Datasets × Models', fontsize=13)

winner_label = comparison_df.iloc[0]['Label']
for ax, metric in zip(axes, ['Test MAE', 'Test RMSE', 'Test R²']):
    colors = ['gold' if lbl == winner_label else 'steelblue'
              for lbl in comparison_df['Label']]
    sns.barplot(ax=ax, data=comparison_df, x='Label', y=metric, palette=colors)
    ax.set_title(metric)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=45, labelsize=7)

plt.tight_layout()
plt.show()

# ── Announce overall winner ───────────────────────────────────────────────────
winner_row   = comparison_df.iloc[0]
best_dataset = winner_row['Dataset']
best_name    = winner_row['Model']

print(f'\n>>> OVERALL WINNER')
print(f'    Dataset  : {best_dataset}')
print(f'    Model    : {best_name}')
print(f'    Test RMSE: {winner_row["Test RMSE"]}  |  Test R²: {winner_row["Test R²"]}')

# ── Expose winner objects for downstream cells ────────────────────────────────
best_res      = all_results[best_dataset][best_name]
best_pipeline = best_res['pipeline']
X_train       = best_res['X_train']
X_test        = best_res['X_test']
y_train       = best_res['y_train']
y_test        = best_res['y_test']
y_pred        = best_res['y_pred']
feature_names = best_res['feature_names']
numeric_cols  = best_res['numeric_cols']
cat_cols      = best_res['cat_cols']

print(f'\nSections 7–10 will use: dataset={best_dataset}, model={best_name}')


## 7. Residual Analysis — Best Pipeline

Using the winning **(dataset, model)** pair identified above.


In [ ]:
resid = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Residual Analysis — {best_dataset}  /  {best_name}', fontsize=12)

sns.scatterplot(ax=axes[0], x=y_pred, y=resid, alpha=0.4)
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Residual')
axes[0].set_title('Predicted vs Residual')

axes[1].hist(resid, bins=40, edgecolor='k', alpha=0.6)
axes[1].set_title('Residual distribution')
axes[1].set_xlabel('Residual')

plt.tight_layout()
plt.show()

print(f'Residual stats: mean={resid.mean():.4f}  std={resid.std():.4f}')


## 8. Feature Importance & SHAP — Best Pipeline


In [ ]:
model_step   = best_pipeline.named_steps['model']
title_suffix = f'{best_dataset}  /  {best_name}'

# ── Tree models: feature importances ─────────────────────────────────────────
if hasattr(model_step, 'feature_importances_'):
    importances = (
        pd.Series(model_step.feature_importances_, index=feature_names)
        .sort_values(ascending=False)
    )
    print('Top 20 feature importances:')
    display(importances.head(20))

    plt.figure(figsize=(10, 6))
    sns.barplot(x=importances.head(20).values, y=importances.head(20).index)
    plt.title(f'Top 20 Feature Importances — {title_suffix}')
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.show()

# ── Linear models: coefficients ───────────────────────────────────────────────
elif hasattr(model_step, 'coef_'):
    coef     = pd.Series(model_step.coef_, index=feature_names)
    coef_abs = coef.abs().sort_values(ascending=False)
    top25    = coef.loc[coef_abs.head(25).index].sort_values()

    print('Top 25 features by absolute coefficient:')
    display(coef.loc[coef_abs.head(25).index])

    colors = ['tomato' if v < 0 else 'steelblue' for v in top25]
    plt.figure(figsize=(10, 8))
    sns.barplot(x=top25.values, y=top25.index, palette=colors)
    plt.axvline(0, color='black', linewidth=0.8)
    plt.title(f'Top 25 Feature Coefficients — {title_suffix}')
    plt.xlabel('Coefficient value')
    plt.tight_layout()
    plt.show()

else:
    print(f'No feature importance or coefficients for {best_name}.')

# ── SHAP — sklearn transform requires pandas ──────────────────────────────────
if shap is not None:
    print('\nRunning SHAP (may take a moment)...')
    X_train_t   = best_pipeline.named_steps['pre'].transform(X_train.to_pandas())
    X_test_t    = best_pipeline.named_steps['pre'].transform(X_test.to_pandas())
    explainer   = shap.Explainer(model_step, X_train_t)
    shap_values = explainer(X_test_t)

    shap.summary_plot(shap_values, feature_names=feature_names, plot_type='bar', show=True)
    shap.summary_plot(shap_values, feature_names=feature_names, show=True)
else:
    print('SHAP not available — restart kernel and re-run to activate.')


## 9. Dummy Patient Prediction — Best Pipeline

Simulate a median/mode patient from the winning dataset's training data, predict their health gain, and explain the key drivers.


In [ ]:
print(f'Dummy patient built from training set: {best_dataset}  /  {best_name}\n')

# ── Build representative patient using Polars native operations ───────────────
dummy_data = {}
for col in numeric_cols:
    dummy_data[col] = [X_train[col].median()]          # Polars returns a scalar
for col in cat_cols:
    mode_series = X_train[col].mode()
    dummy_data[col] = [mode_series[0] if len(mode_series) > 0 else None]

# Keep as Polars DataFrame for consistency.
# Convert to pandas only at the sklearn boundary.
dummy_pl = pl.DataFrame(dummy_data)
dummy_pd = dummy_pl.to_pandas()      # sklearn boundary

print('=== Dummy Patient Profile ===')
display(dummy_pl.transpose(include_header=True, header_name='Feature', column_names=['Value']))

# ── Predict (sklearn boundary) ────────────────────────────────────────────────
predicted_gain = best_pipeline.predict(dummy_pd)[0]
avg_gain       = float(np.mean(y_train))
diff           = predicted_gain - avg_gain
direction      = 'ABOVE' if diff > 0 else 'BELOW'

print(f'\n>>> Predicted Health Gain : {predicted_gain:.4f}')
print(f'    Training set mean     : {avg_gain:.4f}')
print(f'    This patient is {direction} average by {abs(diff):.4f} points\n')

# ── Explain contributions ─────────────────────────────────────────────────────
model_step = best_pipeline.named_steps['model']

if hasattr(model_step, 'coef_'):
    transformed   = best_pipeline.named_steps['pre'].transform(dummy_pd)[0]
    contributions = pd.Series(model_step.coef_ * transformed, index=feature_names)
    top_pos = contributions.sort_values(ascending=False).head(5)
    top_neg = contributions.sort_values().head(5)

    print('=== Top 5 features INCREASING the predicted health gain ===')
    for feat, val in top_pos.items():
        raw = dummy_pl[feat][0] if feat in dummy_pl.columns else 'encoded'
        print(f'  + {feat:<45} raw={raw!r:<10}  contribution={val:+.4f}')

    print('\n=== Top 5 features DECREASING the predicted health gain ===')
    for feat, val in top_neg.items():
        raw = dummy_pl[feat][0] if feat in dummy_pl.columns else 'encoded'
        print(f'  - {feat:<45} raw={raw!r:<10}  contribution={val:+.4f}')

    sorted_contrib = contributions.abs().sort_values(ascending=False).head(20)
    colors = ['steelblue' if contributions[f] >= 0 else 'tomato' for f in sorted_contrib.index]
    plt.figure(figsize=(10, 5))
    sns.barplot(x=contributions[sorted_contrib.index].values,
                y=sorted_contrib.index, palette=colors)
    plt.axvline(0, color='black', linewidth=0.8)
    plt.title(f'Feature Contributions — {best_dataset}  /  {best_name}')
    plt.xlabel('Contribution  (coef × scaled value)')
    plt.tight_layout()
    plt.show()

elif hasattr(model_step, 'feature_importances_'):
    importances = pd.Series(model_step.feature_importances_, index=feature_names).sort_values(ascending=False)
    print('=== Top 10 most important features ===')
    for feat, imp in importances.head(10).items():
        raw = dummy_pl[feat][0] if feat in dummy_pl.columns else 'encoded'
        print(f'  {feat:<45} patient_value={raw!r:<10}  importance={imp:.4f}')

# ── SHAP waterfall (sklearn boundary: .to_pandas()) ───────────────────────────
if shap is not None:
    X_train_t = best_pipeline.named_steps['pre'].transform(X_train.to_pandas())
    X_dummy_t = best_pipeline.named_steps['pre'].transform(dummy_pd)
    explainer = shap.Explainer(model_step, X_train_t)
    shap_vals = explainer(X_dummy_t)
    print('\n=== SHAP Waterfall (dummy patient) ===')
    shap.plots.waterfall(shap_vals[0], max_display=15, show=True)
else:
    print('\nTip: restart kernel and re-run to enable SHAP waterfall.')

print('\n=== Plain English Summary ===')
print(f"Dataset '{best_dataset}' with model '{best_name}' predicts this patient will")
print(f"gain {predicted_gain:.2f} points on the knee health score after surgery.")
print(f"This is {'higher' if diff > 0 else 'lower'} than the average gain of {avg_gain:.2f} in that training set.")


## 10. Save All Models and Metrics

Save the best pipeline for **each dataset** and a combined comparison CSV.


In [ ]:
out_dir = Path('./models')
out_dir.mkdir(parents=True, exist_ok=True)

# ── Save every (dataset × model) pipeline ────────────────────────────────────
for ds_label, models_dict in all_results.items():
    for mdl_name, res in models_dict.items():
        safe_label = ds_label.replace('/', '-').replace(' ', '_')
        pipe_path  = out_dir / f'3.1-{safe_label}-{mdl_name}.joblib'
        joblib.dump(res['pipeline'], pipe_path)
        print(f'Saved: {pipe_path}')

# ── Save comparison CSV ───────────────────────────────────────────────────────
comparison_path = out_dir / '3.1-comparison-metrics.csv'
comparison_df.to_csv(comparison_path, index=False)
print(f'\nSaved comparison table: {comparison_path}')

# ── Print final winner ────────────────────────────────────────────────────────
print(f'\n{"=" * 50}')
print(f'  OVERALL WINNER')
print(f'  Dataset  : {best_dataset}')
print(f'  Model    : {best_name}')
print(f'  Test RMSE: {winner_row["Test RMSE"]}  |  R²: {winner_row["Test R²"]}')
print(f'{"=" * 50}')
